# Análisis de resultados — clasificación dialectal (900 oraciones)

**Notebook de análisis reproducible, sin inferencias.** Trabaja exclusivamente con los cuatro CSV finales generados por `notebooks/experimentos.ipynb`.

- **Clasificación:** Accuracy, Precision, Recall, F1 macro, matrices de confusión y resultados por regla.
- **Recuperación:** regla correcta en frases andinas y abstención correcta en frases no andinas (máximo una regla).
- **Estadística:** McNemar exacto pareado (seis comparaciones, corrección de Holm), intervalos percentiles bootstrap del 95 % (5000 muestras, pareado y estratificado).
- **Errores:** casos discordantes y análisis del fenómeno R05.

**Importante:** la evaluación se refiere a 900 oraciones **sintéticas**. Las predicciones de SUT 1 son *out-of-fold* (validación cruzada estratificada de cinco particiones), mientras que las de SUT 2–4 provienen de inferencia directa. El umbral semántico definitivo usado en SUT 3 es **0.4**, revisado después de una evaluación inicial, por lo que esta configuración tiene carácter exploratorio. El notebook no entrena ni reconfigura ningún sistema, ni evalúa automáticamente la fidelidad lingüística de las explicaciones.

**Uso:** en Colab, elegir **CPU** y ejecutar **Entorno de ejecución → Ejecutar todas**. La carpeta `resultados/analisis_final/` se crea para los archivos derivados sin sobrescribir los CSV de predicciones ni análisis anteriores.

## 1. Conectar Google Drive y preparar bibliotecas

Las bibliotecas utilizadas son las disponibles normalmente en Colab (`pandas`, `numpy`, `scikit-learn`, `scipy` y `matplotlib`). No se descargan modelos de lenguaje ni se necesita GPU.

In [ ]:
from google.colab import drive
from pathlib import Path
import json
import platform
from itertools import combinations
from importlib.metadata import version
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_recall_fscore_support, recall_score,
)

drive.mount('/content/drive')
RAIZ = Path('/content/drive/MyDrive/tesis_experimento')
DATOS = RAIZ / 'datos'
RESULTADOS = RAIZ / 'resultados'
SALIDA = RESULTADOS / 'analisis_final'
SALIDA.mkdir(parents=True, exist_ok=True)

ETIQUETAS = ['andino', 'no_andino']
REGLAS = [f'R{i:02d}' for i in range(1, 10)]
NOMBRES = {
    'SUT 1': 'SUT 1 - Regresión logística',
    'SUT 2': 'SUT 2 - LLM nativo',
    'SUT 3': 'SUT 3 - RAG semántico',
    'SUT 4': 'SUT 4 - RecMS',
}
ARCHIVOS = {
    'SUT 1': 'sut1_ml_predicciones.csv',
    'SUT 2': 'sut2_llm_predicciones.csv',
    'SUT 3': 'sut3_rag_predicciones.csv',
    'SUT 4': 'sut4_recms_predicciones.csv',
}
print('Carpeta de salida:', SALIDA)
print('GPU necesaria: no')

## 2. Validar el conjunto de datos y las 3600 predicciones

Se emparejan los CSV por `case_id`, evitando asumir que todas las filas están originalmente en el mismo orden. Se comprueban textos, etiquetas y agrupaciones, y se rechaza una mezcla accidental de versiones del SUT 3.

In [ ]:
datos = pd.read_csv(DATOS / 'ejemplos_dialectos_900.csv', encoding='utf-8-sig')
assert len(datos) == 900 and datos['case_id'].is_unique
assert set(datos['label']) == set(ETIQUETAS)
assert set(datos['regla_id']) == set(REGLAS)
assert datos.groupby(['regla_id', 'label']).size().eq(50).all()

umbral = float((DATOS / 'umbral_rag.txt').read_text(encoding='utf-8').strip())
assert np.isclose(umbral, 0.4), f'Umbral inesperado: {umbral}'

suts = {}
for sistema, archivo in ARCHIVOS.items():
    df = pd.read_csv(RESULTADOS / archivo, encoding='utf-8-sig')
    assert len(df) == 900 and df['case_id'].is_unique, sistema
    assert set(df['prediccion']) <= set(ETIQUETAS), sistema
    assert df['etiqueta_real'].isin(ETIQUETAS).all(), sistema
    df = df.set_index('case_id').loc[datos['case_id']].reset_index()
    assert df['texto'].tolist() == datos['test_case'].tolist(), sistema
    assert df['etiqueta_real'].tolist() == datos['label'].tolist(), sistema
    assert df['regla_id'].tolist() == datos['regla_id'].tolist(), sistema
    suts[sistema] = df

def normalizar_reglas(columna):
    return (columna.astype('string').str.strip()
            .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, 'null': pd.NA}))

for sistema in ('SUT 3', 'SUT 4'):
    suts[sistema]['regla_recuperada'] = normalizar_reglas(
        suts[sistema]['regla_recuperada']
    )
    assert suts[sistema]['regla_recuperada'].dropna().isin(REGLAS).all(), sistema

# Comprobar que las recuperaciones de SUT 3 cumplen el umbral final.
rag = suts['SUT 3']
score = pd.to_numeric(rag['score_recuperacion'], errors='coerce')
assert score.notna().all()
assert (rag['regla_recuperada'].notna() == score.ge(umbral)).all()

aciertos_esperados = [688, 636, 608, 808]
aciertos_reales = [int(df['prediccion'].eq(df['etiqueta_real']).sum())
                   for df in suts.values()]
assert aciertos_reales == aciertos_esperados, (
    f'Los CSV no coinciden con la corrida final: {aciertos_reales}'
)
print('=== VALIDACIÓN FINAL ===')
for (nombre, _), aciertos in zip(suts.items(), aciertos_reales):
    print(f'{nombre}: 900 casos, {aciertos} aciertos')
print('Total: 3600 predicciones | umbral RAG:', umbral)

## 3. Evaluación global, por clase y matrices de confusión

En la matriz de confusión, las **filas** son etiquetas reales y las **columnas** son predicciones. Todos los valores se obtienen otra vez a partir de los CSV originales.

In [ ]:
filas_global = []
filas_clase = []
filas_matriz = []
matrices = {}

for nombre, df in suts.items():
    real, pred = df['etiqueta_real'], df['prediccion']
    precision, recall, f1, soporte = precision_recall_fscore_support(
        real, pred, labels=ETIQUETAS, zero_division=0
    )
    correctos = int(real.eq(pred).sum())
    filas_global.append({
        'sistema': NOMBRES[nombre], 'accuracy': accuracy_score(real, pred),
        'precision_macro': float(np.mean(precision)),
        'recall_macro': float(np.mean(recall)),
        'f1_macro': float(np.mean(f1)),
        'aciertos': correctos, 'errores': 900-correctos,
    })
    for i, etiqueta in enumerate(ETIQUETAS):
        filas_clase.append({
            'sistema': NOMBRES[nombre], 'clase': etiqueta,
            'precision': precision[i], 'recall': recall[i],
            'f1': f1[i], 'soporte': int(soporte[i]),
        })
    mat = confusion_matrix(real, pred, labels=ETIQUETAS)
    matrices[nombre] = mat
    for i, real_label in enumerate(ETIQUETAS):
        for j, pred_label in enumerate(ETIQUETAS):
            filas_matriz.append({
                'sistema': NOMBRES[nombre], 'clase_real': real_label,
                'clase_predicha': pred_label, 'cantidad': int(mat[i,j]),
            })

tabla_global = pd.DataFrame(filas_global)
tabla_clase = pd.DataFrame(filas_clase)
tabla_matriz = pd.DataFrame(filas_matriz)

tabla_global.to_csv(SALIDA / 'resumen_clasificacion_900.csv', index=False)
tabla_clase.to_csv(SALIDA / 'metricas_por_clase_900.csv', index=False)
tabla_matriz.to_csv(SALIDA / 'matrices_confusion_900.csv', index=False)

print('=== MÉTRICAS GLOBALES ===')
print(tabla_global.round(4).to_string(index=False))
print('\n=== MÉTRICAS POR CLASE ===')
print(tabla_clase.round(4).to_string(index=False))

### 3.1. Rendimiento de SUT 1 en los cinco folds

Además de la métrica calculada sobre las 900 predicciones *out-of-fold*, se informa el F1 macro individual por fold, su media y desviación estándar muestral. No es necesario entrenar nuevamente el clasificador.

In [ ]:
from sklearn.metrics import f1_score
ml = suts['SUT 1']
assert ml['fold'].notna().all()
filas_folds = []
for fold, grupo in ml.groupby('fold', sort=True):
    assert len(grupo) == 180
    assert grupo.groupby(['regla_id','etiqueta_real']).size().eq(10).all()
    filas_folds.append({
        'fold': int(fold), 'n': len(grupo),
        'f1_macro': f1_score(grupo['etiqueta_real'], grupo['prediccion'],
                             labels=ETIQUETAS, average='macro', zero_division=0)
    })
tabla_folds = pd.DataFrame(filas_folds)
assert len(tabla_folds) == 5
f1_cv_media = float(tabla_folds['f1_macro'].mean())
f1_cv_sd = float(tabla_folds['f1_macro'].std(ddof=1))
tabla_folds.to_csv(SALIDA / 'f1_sut1_por_fold.csv', index=False)
print('=== SUT 1: F1 MACRO POR FOLD ===')
print(tabla_folds.round(4).to_string(index=False))
print(f'Media ± DE: {f1_cv_media:.4f} ± {f1_cv_sd:.4f}')
print('F1 macro consolidado:', round(float(tabla_global.loc[0,'f1_macro']),4))

In [ ]:
fig, ejes = plt.subplots(2, 2, figsize=(10, 8), constrained_layout=True)
for eje, (nombre, matriz) in zip(ejes.flat, matrices.items()):
    im = eje.imshow(matriz, cmap='Blues', vmin=0, vmax=450)
    eje.set_title(nombre)
    eje.set_xlabel('Predicción')
    eje.set_ylabel('Etiqueta real')
    eje.set_xticks(range(2), ETIQUETAS)
    eje.set_yticks(range(2), ETIQUETAS)
    for i in range(2):
        for j in range(2):
            eje.text(j, i, str(matriz[i,j]), ha='center', va='center',
                     color='white' if matriz[i,j] > 225 else 'black', fontsize=13)
fig.savefig(SALIDA / 'matrices_confusion_900.png', dpi=200, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(8, 4.5))
valores = tabla_global['f1_macro'].to_numpy()
barras = ax.bar(list(NOMBRES), valores)
ax.set_ylim(0, 1)
ax.set_ylabel('F1 macro')
ax.set_title('Clasificación: 900 oraciones sintéticas')
for barra, valor in zip(barras, valores):
    ax.text(barra.get_x()+barra.get_width()/2, valor+0.015,
            f'{valor:.4f}', ha='center')
fig.tight_layout()
fig.savefig(SALIDA / 'f1_macro_global_900.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Clasificación por regla lingüística

Cada regla contiene 50 ejemplos andinos y 50 no andinos. Se reportan F1 macro y Recall por clase, y se calcula la diferencia de F1 entre SUT 4 y SUT 3.

In [ ]:
filas = []
for nombre, df in suts.items():
    for regla in REGLAS:
        grupo = df[df['regla_id'].eq(regla)]
        assert len(grupo) == 100
        real, pred = grupo['etiqueta_real'], grupo['prediccion']
        filas.append({
            'sistema': nombre, 'regla_id': regla,
            'accuracy': accuracy_score(real, pred),
            'f1_macro': f1_score(real, pred, labels=ETIQUETAS,
                                 average='macro', zero_division=0),
            'recall_andino': recall_score(real, pred, pos_label='andino'),
            'recall_no_andino': recall_score(real, pred, pos_label='no_andino'),
            'aciertos': int(real.eq(pred).sum()),
            'errores': int(real.ne(pred).sum()),
        })

por_regla = pd.DataFrame(filas)
f1_reglas = por_regla.pivot(index='regla_id', columns='sistema', values='f1_macro')
f1_reglas = f1_reglas.reindex(index=REGLAS, columns=list(NOMBRES))
recall_andino = por_regla.pivot(index='regla_id', columns='sistema', values='recall_andino')
recall_andino = recall_andino.reindex(index=REGLAS, columns=list(NOMBRES))

diferencias = pd.DataFrame({
    'regla_id': REGLAS,
    'f1_sut3': f1_reglas['SUT 3'].to_numpy(),
    'f1_sut4': f1_reglas['SUT 4'].to_numpy(),
})
diferencias['diferencia'] = diferencias['f1_sut4'] - diferencias['f1_sut3']

por_regla.to_csv(SALIDA / 'metricas_clasificacion_por_regla_900.csv', index=False)
diferencias.to_csv(SALIDA / 'diferencias_f1_por_regla_900.csv', index=False)
print('=== F1 MACRO POR REGLA ===')
print(f1_reglas.round(4).to_string())
print('\n=== RECALL ANDINO POR REGLA ===')
print(recall_andino.round(4).to_string())
print('\n=== DIFERENCIA DE F1: SUT 4 - SUT 3 ===')
print(diferencias.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
imagen = ax.imshow(f1_reglas.to_numpy(), vmin=0, vmax=1,
                   cmap='viridis', aspect='auto')
ax.set_xticks(range(4), f1_reglas.columns)
ax.set_yticks(range(9), f1_reglas.index)
ax.set_xlabel('Sistema')
ax.set_ylabel('Regla lingüística')
ax.set_title('F1 macro por regla y sistema')
for i in range(9):
    for j in range(4):
        valor = f1_reglas.iloc[i, j]
        ax.text(j, i, f'{valor:.3f}', ha='center', va='center',
                color='white' if valor < 0.5 else 'black')
fig.colorbar(imagen, ax=ax, label='F1 macro')
fig.tight_layout()
fig.savefig(SALIDA / 'f1_macro_por_regla_900.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Recuperación de reglas con abstención

**Definición operacional del experimento:** una oración `andino` acierta si se recupera su `regla_id`; una oración `no_andino` acierta cuando **no** se recupera ninguna regla. El `regla_id` de los ejemplos no andinos es metadato de agrupación, no una regla que el recuperador deba recuperar.

El nombre «Hit Rate@1 con abstención» distingue esta métrica de una tasa de acierto top-1 convencional; las tasas por clase se presentan por separado.

In [ ]:
filas_rec = []
filas_regla_rec = []
for nombre in ('SUT 3', 'SUT 4'):
    df = suts[nombre]
    for regla in REGLAS:
        grupo = df[df['regla_id'].eq(regla)]
        andinas = grupo[grupo['etiqueta_real'].eq('andino')]
        no_andinas = grupo[grupo['etiqueta_real'].eq('no_andino')]
        assert len(andinas) == len(no_andinas) == 50

        rec_a = andinas['regla_recuperada']
        rec_n = no_andinas['regla_recuperada']
        correctas = int(rec_a.eq(regla).fillna(False).sum())
        incorrectas = int((rec_a.notna() & ~rec_a.eq(regla).fillna(False)).sum())
        faltantes = int(rec_a.isna().sum())
        abstenciones = int(rec_n.isna().sum())
        indebidas = int(rec_n.notna().sum())
        assert correctas + incorrectas + faltantes == 50
        assert abstenciones + indebidas == 50

        filas_regla_rec.append({
            'sistema': nombre, 'regla_id': regla,
            'correctas_andino': correctas, 'incorrectas_andino': incorrectas,
            'sin_regla_andino': faltantes,
            'abstenciones_no_andino': abstenciones,
            'recuperaciones_indebidas': indebidas,
            'hit_andino': correctas/50,
            'abstencion_no_andino': abstenciones/50,
            'hit_global': (correctas+abstenciones)/100,
        })

    sub = pd.DataFrame(filas_regla_rec)
    sub = sub[sub['sistema'].eq(nombre)]
    ca, ia, sa = [int(sub[c].sum()) for c in
                  ['correctas_andino', 'incorrectas_andino', 'sin_regla_andino']]
    an, rn = [int(sub[c].sum()) for c in
              ['abstenciones_no_andino', 'recuperaciones_indebidas']]
    filas_rec.append({
        'sistema': NOMBRES[nombre],
        'hit_at_1_abstencion': (ca+an)/900,
        'reglas_correctas_andino': ca, 'reglas_incorrectas_andino': ia,
        'sin_regla_andino': sa,
        'abstenciones_correctas_no_andino': an,
        'reglas_indebidas_no_andino': rn,
        'recall_regla_andino': ca/450,
        'abstencion_no_andino': an/450,
    })

tabla_rec = pd.DataFrame(filas_rec)
rec_por_regla = pd.DataFrame(filas_regla_rec)
assert tabla_rec[['reglas_correctas_andino',
                  'abstenciones_correctas_no_andino']].values.tolist() == [
                      [115, 223], [403, 443]]

tabla_rec.to_csv(SALIDA / 'comparacion_recuperacion_900.csv', index=False)
rec_por_regla.to_csv(SALIDA / 'recuperacion_por_regla_900.csv', index=False)

print('=== RECUPERACIÓN GLOBAL ===')
print(tabla_rec.round(4).to_string(index=False))
for nombre_col, titulo in [
    ('hit_andino', 'RECUPERACIÓN CORRECTA EN ANDINAS'),
    ('abstencion_no_andino', 'ABSTENCIÓN EN NO ANDINAS'),
    ('hit_global', 'HIT@1 CON ABSTENCIÓN POR REGLA'),
]:
    print(f'\n=== {titulo} ===')
    print(rec_por_regla.pivot(index='regla_id', columns='sistema',
                               values=nombre_col).round(4).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
valores = tabla_rec['hit_at_1_abstencion'].to_numpy()
barras = ax.bar(['SUT 3', 'SUT 4'], valores)
ax.set_ylim(0, 1)
ax.set_ylabel('Proporción de aciertos')
ax.set_title('Recuperación correcta y abstención (900 casos)')
for barra, valor in zip(barras, valores):
    ax.text(barra.get_x()+barra.get_width()/2, valor+0.018,
            f'{valor:.4f}', ha='center')
fig.tight_layout()
fig.savefig(SALIDA / 'recuperacion_con_abstencion_900.png',
            dpi=200, bbox_inches='tight')
plt.show()

## 6. Comparaciones pareadas: McNemar exacto y corrección de Holm

La prueba exacta de McNemar compara **aciertos/errores pareados**, no F1 macro. Se realizan seis contrastes y se ajustan sus valores p mediante Holm. La comparación central es SUT 3 frente a SUT 4.

In [ ]:
aciertos = pd.DataFrame({'case_id': datos['case_id']})
for nombre, df in suts.items():
    aciertos[nombre] = df['prediccion'].eq(df['etiqueta_real']).to_numpy()

filas_mcnemar = []
for a, b in combinations(ARCHIVOS, 2):
    ca = aciertos[a].to_numpy()
    cb = aciertos[b].to_numpy()
    ambos = int((ca & cb).sum())
    solo_a = int((ca & ~cb).sum())
    solo_b = int((~ca & cb).sum())
    ninguno = int((~ca & ~cb).sum())
    n_disc = solo_a + solo_b
    p = float(binomtest(solo_a, n_disc, p=0.5).pvalue) if n_disc else 1.0
    filas_mcnemar.append({
        'sistema_a': a, 'sistema_b': b,
        'ambos_correctos': ambos, 'solo_a_correcto': solo_a,
        'solo_b_correcto': solo_b, 'ambos_incorrectos': ninguno,
        'discordantes': n_disc,
        'diferencia_accuracy_b_menos_a': (solo_b-solo_a)/900,
        'p_valor': p,
    })

tabla_mcnemar = pd.DataFrame(filas_mcnemar).sort_values('p_valor').reset_index(drop=True)
m = len(tabla_mcnemar)
p_ajustados = np.maximum.accumulate(
    tabla_mcnemar['p_valor'].to_numpy() * np.arange(m, 0, -1)
)
tabla_mcnemar['p_holm'] = np.minimum(p_ajustados, 1.0)
tabla_mcnemar['significativo_holm'] = tabla_mcnemar['p_holm'] < 0.05

tabla_mcnemar.to_csv(SALIDA / 'mcnemar_pares_holm_900.csv', index=False)

principal_m = tabla_mcnemar[
    tabla_mcnemar['sistema_a'].eq('SUT 3') &
    tabla_mcnemar['sistema_b'].eq('SUT 4')
].iloc[0]
assert [int(principal_m[c]) for c in [
    'ambos_correctos', 'solo_a_correcto', 'solo_b_correcto',
    'ambos_incorrectos']] == [590, 18, 218, 74]

tabla_mcnemar[
    tabla_mcnemar['sistema_a'].eq('SUT 3') &
    tabla_mcnemar['sistema_b'].eq('SUT 4')
].to_csv(SALIDA / 'mcnemar_sut3_vs_sut4.csv', index=False)

print('=== MCNEMAR EXACTO + HOLM ===')
print(tabla_mcnemar.to_string(index=False,
    float_format=lambda v: f'{v:.6g}'))

## 7. Intervalos de confianza: bootstrap pareado y estratificado

5000 remuestreos con reemplazo, manteniendo 450 oraciones por clase y usando los **mismos índices** para todos los SUT. Se calculan intervalos percentiles del 95 % para la **diferencia de Accuracy y de F1 macro**. Estos intervalos cuantifican la variabilidad condicional a los casos sintéticos y no representan incertidumbre del entrenamiento ni garantizan generalización a otros textos.

In [ ]:
B = 5000
rng = np.random.default_rng(42)
y = datos['label'].eq('andino').to_numpy()
pred = np.stack([suts[n]['prediccion'].eq('andino').to_numpy()
                 for n in ARCHIVOS])
idx_a = np.flatnonzero(y)
idx_n = np.flatnonzero(~y)
assert len(idx_a) == len(idx_n) == 450

muest_a = rng.choice(idx_a, size=(B, 450), replace=True)
muest_n = rng.choice(idx_n, size=(B, 450), replace=True)
TP = pred[:, muest_a].sum(axis=2)
FN = 450 - TP
FP = pred[:, muest_n].sum(axis=2)
TN = 450 - FP
acc_boot = (TP + TN) / 900
f1_boot = 0.5 * (
    (2 * TP)/(2 * TP + FP + FN) +
    (2 * TN)/(2 * TN + FP + FN)
)

acc_punto = tabla_global['accuracy'].to_numpy()
f1_punto = tabla_global['f1_macro'].to_numpy()
filas_ic = []
for i, j in combinations(range(4), 2):
    da = acc_boot[j] - acc_boot[i]
    df = f1_boot[j] - f1_boot[i]
    ic_a = np.quantile(da, [.025, .975])
    ic_f = np.quantile(df, [.025, .975])
    filas_ic.append({
        'sistema_a': list(ARCHIVOS)[i],
        'sistema_b': list(ARCHIVOS)[j],
        'diferencia_accuracy': acc_punto[j] - acc_punto[i],
        'ic95_accuracy_inferior': ic_a[0],
        'ic95_accuracy_superior': ic_a[1],
        'diferencia_f1_macro': f1_punto[j] - f1_punto[i],
        'ic95_f1_inferior': ic_f[0],
        'ic95_f1_superior': ic_f[1],
    })

tabla_ic = pd.DataFrame(filas_ic)
tabla_ic.to_csv(SALIDA / 'intervalos_confianza_pareados_900.csv', index=False)
principal_ic = tabla_ic[
    tabla_ic['sistema_a'].eq('SUT 3') &
    tabla_ic['sistema_b'].eq('SUT 4')
].iloc[0]
print('=== INTERVALOS DE CONFIANZA, 5000 REMUESTREOS ===')
print(tabla_ic.to_string(index=False, float_format=lambda v: f'{v:.4f}'))
print('\nSUT 4 - SUT 3 | Accuracy: {:.4f} [{:.4f}, {:.4f}]'.format(
    principal_ic['diferencia_accuracy'],
    principal_ic['ic95_accuracy_inferior'],
    principal_ic['ic95_accuracy_superior']))
print('SUT 4 - SUT 3 | F1 macro: {:.4f} [{:.4f}, {:.4f}]'.format(
    principal_ic['diferencia_f1_macro'],
    principal_ic['ic95_f1_inferior'],
    principal_ic['ic95_f1_superior']))

## 8. Análisis cualitativo de errores y diagnóstico de R05

Se conservan las explicaciones textuales de ambos RAG y se identifica si la regla fue correcta, equivocada o no recuperada. Para R05 se separa explícitamente **recuperación** de **clasificación**. Esta sección prepara ejemplos para examen lingüístico; no asigna puntuaciones automáticas de fidelidad.

In [ ]:
casos = pd.DataFrame({
    'case_id': datos['case_id'], 'regla_id': datos['regla_id'],
    'texto': datos['test_case'], 'etiqueta_real': datos['label'],
})
for i, nombre in enumerate(ARCHIVOS, 1):
    casos[f'pred_sut{i}'] = suts[nombre]['prediccion']
for i in (3, 4):
    df = suts[f'SUT {i}']
    casos[f'regla_sut{i}'] = df['regla_recuperada']
    casos[f'explicacion_sut{i}'] = df['explicacion']

c3 = casos['pred_sut3'].eq(casos['etiqueta_real'])
c4 = casos['pred_sut4'].eq(casos['etiqueta_real'])
casos['categoria'] = np.select(
    [c3 & c4, ~c3 & c4, c3 & ~c4, ~c3 & ~c4],
    ['Ambos correctos', 'Solo RecMS correcto', 'Solo RAG correcto',
     'Ambos incorrectos'], default='Sin categoría',
)
assert casos['categoria'].value_counts().to_dict() == {
    'Ambos correctos': 590,
    'Solo RecMS correcto': 218,
    'Ambos incorrectos': 74,
    'Solo RAG correcto': 18,
}

for i in (3, 4):
    recuperada = casos[f'regla_sut{i}']
    andina = casos['etiqueta_real'].eq('andino')
    correcta = recuperada.eq(casos['regla_id']).fillna(False)
    casos[f'estado_rec_sut{i}'] = np.select(
        [andina & correcta, andina & recuperada.isna(),
         andina & recuperada.notna() & ~correcta,
         ~andina & recuperada.isna(),
         ~andina & recuperada.notna()],
        ['Regla correcta', 'Sin regla', 'Regla equivocada',
         'Abstención correcta', 'Recuperación indebida'],
        default='Sin categoría',
    )

casos.to_csv(SALIDA / 'analisis_final_casos_900.csv', index=False)
casos[casos['categoria'].ne('Ambos correctos')].to_csv(
    SALIDA / 'casos_con_error_sut3_o_sut4.csv', index=False
)

resumen_errores = pd.crosstab(
    [casos['regla_id'], casos['etiqueta_real']], casos['categoria']
).reset_index()
resumen_errores.to_csv(SALIDA / 'errores_por_regla_y_clase.csv', index=False)

filas_cruces = []
for i in (3, 4):
    temporal = casos.copy()
    temporal['clasificacion_correcta'] = temporal[f'pred_sut{i}'].eq(
        temporal['etiqueta_real'])
    g = (temporal.groupby(
        ['etiqueta_real', f'estado_rec_sut{i}', 'clasificacion_correcta']
    ).size().reset_index(name='cantidad'))
    g = g.rename(columns={f'estado_rec_sut{i}': 'estado_recuperacion'})
    g.insert(0, 'sistema', f'SUT {i}')
    filas_cruces.append(g)
cruces = pd.concat(filas_cruces, ignore_index=True)
cruces.to_csv(SALIDA / 'recuperacion_vs_clasificacion.csv', index=False)

ejemplos = (casos[
    casos['categoria'].isin(['Solo RecMS correcto', 'Solo RAG correcto',
                              'Ambos incorrectos'])
].sort_values('case_id').groupby(
    ['categoria', 'regla_id', 'etiqueta_real'], sort=True
).head(2).reset_index(drop=True))
ejemplos.to_csv(SALIDA / 'ejemplos_para_discusion.csv', index=False)

r05 = casos[casos['regla_id'].eq('R05')].copy()
r05.to_csv(SALIDA / 'analisis_R05.csv', index=False)
r05_a = r05[r05['etiqueta_real'].eq('andino')]
assert len(r05_a) == 50
assert int(r05_a['regla_sut4'].eq('R05').sum()) == 50
assert int(r05_a['pred_sut4'].eq('andino').sum()) == 1

print('=== TIPOS DE ERROR ===')
print(casos['categoria'].value_counts().to_string())
print('\n=== RECUPERACIÓN Y CLASIFICACIÓN ===')
print(cruces.to_string(index=False))
print('\n=== R05 ANDINO, SUT 4 ===')
print('50/50 reglas recuperadas; 1/50 predicciones andinas correctas')
print(r05_a[['case_id','texto','pred_sut4','explicacion_sut4']]
      .head(5).to_string(index=False))

## 9. Consolidar archivos para el repositorio y la redacción

Se genera un resumen en Markdown, un registro básico de bibliotecas y un ZIP con los cuatro CSV originales de predicciones y **todos** los resultados regenerados por este notebook. Las fuentes en `src/` y el dataset se mantienen en sus carpetas originales para publicarlos en GitHub por separado.

In [ ]:
resumen = f'''# Resultados del experimento (900 oraciones sintéticas)

## Configuración

- Clases: andino / no_andino (450 ejemplos por clase).
- Nueve reglas R01–R09, 50 casos por combinación de regla y clase.
- SUT 1: embeddings spaCy + regresión logística; cinco folds estratificados (predicciones fuera del fold de entrenamiento).
- SUT 2: Llama 3.1 8B Instruct sin reglas recuperadas.
- SUT 3: BGE-M3 + FAISS + Llama; recuperación top-1 y umbral coseno 0.4.
- SUT 4: Stanza + detectores heurísticos en orden de prioridad + Llama; recuperación top-1.
- **Nota de protocolo:** el umbral 0.4 de SUT 3 se escogió tras una evaluación inicial (análisis exploratorio).

## Clasificación global

{tabla_global.to_markdown(index=False, floatfmt='.4f')}

## Validación cruzada de SUT 1

- Media del F1 macro de cinco folds: {f1_cv_media:.4f}; DE muestral: {f1_cv_sd:.4f}.

## Recuperación con abstención

{tabla_rec.to_markdown(index=False, floatfmt='.4f')}

La definición operacional valora recuperar la regla objetivo en frases andinas y no recuperar regla en frases no andinas.

## Contrastes pareados

- McNemar exacto SUT 3 vs SUT 4, valor p ajustado de Holm: {principal_m['p_holm']:.6g}.
- Diferencia en accuracy SUT 4 − SUT 3: {principal_ic['diferencia_accuracy']:.4f}; IC95 bootstrap [{principal_ic['ic95_accuracy_inferior']:.4f}, {principal_ic['ic95_accuracy_superior']:.4f}].
- Diferencia en F1 macro SUT 4 − SUT 3: {principal_ic['diferencia_f1_macro']:.4f}; IC95 bootstrap [{principal_ic['ic95_f1_inferior']:.4f}, {principal_ic['ic95_f1_superior']:.4f}].
- Comparaciones por pares: 6; significativas tras Holm: {int(tabla_mcnemar['significativo_holm'].sum())}.

## Caso R05

En las 50 oraciones andinas asignadas a R05, SUT 4 recupera R05 en 50/50 casos, pero clasifica correctamente solo 1/50. La recuperación pertinente no garantiza una etiqueta correcta del LLM.

## Alcance

Resultados del conjunto sintético estudiado. No se presupone generalización a otros materiales dialectales ni evaluación automática de fidelidad de explicaciones.
'''
(SALIDA / 'consolidado_resultados.md').write_text(resumen, encoding='utf-8')

versiones = {
    'python': platform.python_version(),
    'pandas': version('pandas'), 'numpy': version('numpy'),
    'scipy': version('scipy'), 'scikit-learn': version('scikit-learn'),
    'matplotlib': version('matplotlib'),
    'bootstrap_muestras': B, 'bootstrap_semilla': 42,
    'umbral_semantico': umbral,
}
(SALIDA / 'entorno_analisis.json').write_text(
    json.dumps(versiones, ensure_ascii=False, indent=2), encoding='utf-8'
)

paquete = SALIDA / 'paquete_analisis_final.zip'
with ZipFile(paquete, 'w', ZIP_DEFLATED) as z:
    for archivo in ARCHIVOS.values():
        z.write(RESULTADOS / archivo, arcname=f'predicciones/{archivo}')
    for archivo in SALIDA.iterdir():
        if archivo.is_file() and archivo != paquete:
            z.write(archivo, arcname=f'analisis/{archivo.name}')

print('=== ANÁLISIS FINAL COMPLETADO ===')
print('Métricas y figuras:', SALIDA)
print('Archivos generados:', len([p for p in SALIDA.iterdir() if p.is_file()]))
print('Paquete:', paquete)
print('Comparación principal: F1 SUT4-SUT3 = {:.4f}'.format(
    principal_ic['diferencia_f1_macro']))